# Phase 12: SHAP Explainability & Decision Transparency

## 1. Objective
This notebook demonstrates **SHapley Additive exPlanations (SHAP)** applied to the optimized XGBoost multiclass routing classifier (Phase 6), coupled with transparent context-aware decision arbitration from the Phase 11 Smart Routing Engine.

### Scientific Interpretation Mandate
> **CRITICAL SCIENTIFIC DISCLAIMER:**
> SHAP values describe how features influence the trained model's mathematical output space.
> They should **NOT** be interpreted as causal effects, biological mechanisms, or regulatory safety criteria.
> Statements like *"BOD causes Sewer Bypass"* are invalid. The scientifically rigorous phrasing is:
> *"High BOD contributed to the model's Sewer Bypass prediction."*
> Furthermore, because the supervised routing labels are rule-derived operational labels, SHAP primarily explains how the model learned that operational labeling framework rather than independently validated real-world reuse outcomes.


## 2. Load Optimized XGBoost Model
We load the frozen, serialized XGBoost model artifact (`models/xgboost_optimized.pkl`) produced during Phase 6 without retraining or altering weights.


In [ ]:
import sys, os, types
import numpy as np
import pandas as pd
import joblib

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath(".."))

# Windows Application Control compatibility patch
if 'sklearn.metrics.cluster._expected_mutual_info_fast' not in sys.modules:
    m1 = types.ModuleType('sklearn.metrics.cluster._expected_mutual_info_fast')
    m1.expected_mutual_information = lambda *a, **kw: 0.0
    sys.modules['sklearn.metrics.cluster._expected_mutual_info_fast'] = m1

if 'sklearn.linear_model._sgd_fast' not in sys.modules:
    m2 = types.ModuleType('sklearn.linear_model._sgd_fast')
    class DummyLoss: pass
    m2.EpsilonInsensitive = DummyLoss; m2.Hinge = DummyLoss; m2.ModifiedHuber = DummyLoss
    m2.SquaredEpsilonInsensitive = DummyLoss; m2.SquaredHinge = DummyLoss
    m2._plain_sgd32 = lambda *a, **kw: None; m2._plain_sgd64 = lambda *a, **kw: None
    sys.modules['sklearn.linear_model._sgd_fast'] = m2

import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Verify model loading
model_path = os.path.join("..", "models", "xgboost_optimized.pkl")
model = joblib.load(model_path)
print("XGBoost Model loaded successfully:", type(model))
print("Number of boosted rounds:", model.get_booster().num_boosted_rounds())


## 3. Load Preprocessing Pipeline and Feature Alignment
We verify feature names and order against the canonical 19-feature vector (one-hot encoded greywater sources + 15 physical/biological parameters).


In [ ]:
from anomaly.anomaly_detector import EXPECTED_FEATURES
from routing.decision_schema import CLASS_MAP

print("Total features required: " + str(len(EXPECTED_FEATURES)))
for idx, feat_name in enumerate(EXPECTED_FEATURES, start=1):
    print("  %2d. %s" % (idx, feat_name))

print("\nRouting Classes:")
for k, v in CLASS_MAP.items():
    print("  Class %d: %s" % (k, v))


## 4. Prepare Explanation Dataset
We load the training set (`dataset/processed/train.csv`) and test set (`dataset/processed/test.csv`).


In [ ]:
train_path = os.path.join("..", "dataset", "processed", "train.csv")
test_path = os.path.join("..", "dataset", "processed", "test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df[EXPECTED_FEATURES]
y_train = train_df["Routing_Class"]
X_test = test_df[EXPECTED_FEATURES]
y_test = test_df["Routing_Class"]

print("Train features shape:", X_train.shape)
print("Test features shape:", X_test.shape)


## 5. Create TreeExplainer
We instantiate `shap.TreeExplainer` on the optimized XGBoost booster.


In [ ]:
from explainability.shap_explainer import ShapRoutingExplainer

explainer_wrapper = ShapRoutingExplainer(model_path=model_path)
explainer = explainer_wrapper.explainer
print("TreeExplainer initialized:", type(explainer))


## 6. Inspect Multiclass SHAP Output Structure
We compute SHAP explanations for the test set and inspect tensor dimensions.


In [ ]:
exp_test, X_test_formatted = explainer_wrapper.explain_batch(test_df)
print("SHAP values tensor shape (N, features, classes):", exp_test.values.shape)
print("Base values shape:", exp_test.base_values.shape)
print("Feature names in explanation:", len(exp_test.feature_names))


## 7. Global SHAP Feature Importance
Global feature importance is calculated as the mean absolute SHAP value across all samples and classes:
$$\text{GlobalImportance}(f) = \frac{1}{N \cdot C} \sum_{i=1}^N \sum_{c=0}^{C-1} |\phi_{i, c}(f)|$$


In [ ]:
from explainability.global_explainer import GlobalShapExplainer

global_explainer = GlobalShapExplainer(explainer_wrapper)
global_imp = global_explainer.compute_global_importance(train_df)
print("Top 10 Global Features by Mean Absolute SHAP:")
print(global_imp.head(10).to_string(index=False))


## 8. Class-Specific SHAP Importance
Mean absolute SHAP value evaluated per routing class shows which parameters dominate decisions for each respective route.


In [ ]:
class_imp = global_explainer.compute_class_importance(train_df)

for c in range(4):
    c_name = CLASS_MAP[c]
    sub = class_imp[class_imp["class"] == c].head(5)
    print("\n--- Class %d: %s (Top 5 Features) ---" % (c, c_name))
    print(sub[["rank", "feature", "mean_abs_shap"]].to_string(index=False))


## 9. Beeswarm Analysis
The beeswarm plot illustrates feature value distribution (low to high) vs directional contribution to model log-odds.


In [ ]:
# Beeswarm for Class 0 (Sewer Bypass)
plt.figure(figsize=(10, 6), dpi=120)
shap.summary_plot(exp_test.values[:, :, 0], X_test_formatted, show=False)
plt.title("SHAP Beeswarm Summary — Class 0 (Sewer Bypass)", fontsize=12)
plt.tight_layout()
plt.show()


## 10. Local Instance-Level Explanations
Using `LocalShapExplainer`, we extract the top positive, negative, and absolute contributors for individual samples.


In [ ]:
from explainability.local_explainer import LocalShapExplainer

local_explainer = LocalShapExplainer(explainer_wrapper)
sample_1 = test_df.iloc[0].to_dict()
loc_res = local_explainer.explain_instance(sample_1)

print("Predicted Route:", loc_res["predicted_route"])
print("Confidence:", str(round(loc_res["prediction_confidence"] * 100, 2)) + "%")
print("Base Value:", loc_res["base_value"])
print("\nTop 5 Absolute Contributors:")
for f in loc_res["top_absolute_contributors"]:
    print("  %d. %-20s = %10.2f (SHAP: %+7.4f, %s)" % (
        f["importance_rank"], f["feature"], f["value"], f["shap_value"], f["direction"]
    ))

print("\nHuman-Readable Text:")
print(loc_res["explanation_text"])


## 11. Waterfall Examples
Waterfall plots show the step-by-step contribution of each feature from base value $E[f(X)]$ to final margin $f(x)$.


In [ ]:
fig_wf = os.path.join("..", "reports", "figures", "shap", "sample_1_waterfall.png")
if os.path.exists(fig_wf):
    print("Sample 1 Waterfall figure verified at: " + fig_wf)
else:
    local_explainer.generate_waterfall_plot(sample_1, fig_wf)
    print("Generated waterfall figure at: " + fig_wf)


## 12. Prediction vs. Final Routing Separation
Crucial architectural distinction:
- **ML SHAP Explanation**: explains what the trained model computed from water-quality features.
- **Rule-Based Override**: explains why external policies (pathogen cutoff, heavy rainfall, high deterioration) altered the operational dispatch.


In [ ]:
from explainability.decision_explainer import DecisionExplainer

decision_explainer = DecisionExplainer()
explained_batch_path = os.path.join("..", "dataset", "processed", "routing_decisions_explained.csv")
if os.path.exists(explained_batch_path):
    df_exp = pd.read_csv(explained_batch_path)
    overrides = df_exp[df_exp["override_applied"] == True]
    print("Total test instances: " + str(len(df_exp)))
    print("Total overrides applied: " + str(len(overrides)))
    print("\nSample Override Entries:")
    print(overrides[["Sample_ID", "predicted_route", "final_route", "final_action"]].head(5).to_string(index=False))


## 13. Scenario: Critical Safety Override
Demonstrating an acute biological hazard override where the model prediction is strictly separated from the safety rule.


In [ ]:
acute_sample = {
    "Greywater_Source": "Bathroom",
    "pH": 7.3,
    "TEMP_C": 24.0,
    "SAL_ppt": 0.2,
    "TUR_NTU": 35.0,
    "DS_mg_L": 270.0,
    "TDS_mg_L": 290.0,
    "TSS_mg_L": 42.0,
    "COND_uS_cm": 530.0,
    "DO_mg_L": 4.6,
    "BOD_mg_L": 38.0,
    "COD_mg_L": 95.0,
    "NH4F_mg_L": 4.8,
    "NO3_mg_L": 4.2,
    "K_mg_L": 16.0,
    "E_coli_CFU_100mL": 650000.0 # Acute pathogen cutoff
}

res_override = decision_explainer.explain_decision(acute_sample)
print("ML Model Prediction:", res_override["model_prediction"]["route"])
print("Final Route:", res_override["final_decision"]["route"])
print("Final Action:", res_override["final_decision"]["action"])
print("Override Applied:", res_override["final_decision"]["override_applied"])
print("\nExplanation:")
print(res_override["human_readable_explanation"])


## 14. Scenario: Environmental Weather Deferral
When water quality permits Restricted Irrigation but meteorological forecasts indicate heavy rainfall, irrigation is deferred without altering the water-quality assessment.


In [ ]:
irrigation_sample = {
    "Greywater_Source": "Bathroom",
    "pH": 7.4,
    "TEMP_C": 24.0,
    "SAL_ppt": 0.2,
    "TUR_NTU": 25.0,
    "DS_mg_L": 250.0,
    "TDS_mg_L": 270.0,
    "TSS_mg_L": 35.0,
    "COND_uS_cm": 510.0,
    "DO_mg_L": 5.0,
    "BOD_mg_L": 30.0,
    "COD_mg_L": 85.0,
    "NH4F_mg_L": 4.0,
    "NO3_mg_L": 4.5,
    "K_mg_L": 16.0,
    "E_coli_CFU_100mL": 1200.0
}

rainy_weather = {
    "rainfall_mm": 28.0,
    "precipitation_probability": 0.95,
    "temperature_C": 19.0,
    "weather_status": "LIVE"
}

res_weather = decision_explainer.explain_decision(irrigation_sample, override_weather=rainy_weather)
print("ML Model Route:", res_weather["model_prediction"]["route"])
print("Final Route:", res_weather["final_decision"]["route"])
print("Final Action:", res_weather["final_decision"]["action"])
print("\nExplanation:")
print(res_weather["human_readable_explanation"])


## 15. Scenario: Storage Tank Biochemical Deterioration
When storage retention time leads to biological anaerobic decomposition, operational recirculation or sewer discharge is triggered.


In [ ]:
degraded_storage = {
    "deterioration_index": 0.85,
    "stagnation_status": "DEGRADED",
    "storage_age_hours": 48.0
}

res_storage = decision_explainer.explain_decision(irrigation_sample, override_storage=degraded_storage)
print("Storage Status:", res_storage["storage_explanation"]["status"])
print("Deterioration Index:", res_storage["storage_explanation"]["deterioration_index"])
print("Final Action:", res_storage["final_decision"]["action"])
print("\nExplanation:")
print(res_storage["human_readable_explanation"])


## 16. Scientific and Technical Limitations
1. **Mathematical Attribution vs Physical Causation**: SHAP values compute credit allocation in the model's feature space. They do not demonstrate physical biochemical causality.
2. **Operational Framework Artifact**: Because the supervised labels are rule-derived, SHAP explains how the tree ensembles approximated the deterministic labeling criteria.
3. **Feature Correlation / Interdependence**: TreeExplainer handles correlated features by conditioning on tree paths, but high collinearity (e.g. TDS vs Salinity or BOD vs COD) can spread attributions between correlated inputs.
4. **Multiclass Log-Odds Representation**: Multiclass SHAP values are expressed in margin space prior to softmax normalization, requiring careful directional interpretations.


## 17. Conclusions
- **Full Transparency Achieved**: SHAP TreeExplainer delivers mathematically grounded local and global feature attributions without retraining models or modifying datasets.
- **Architectural Separation Maintained**: The decision engine preserves strict boundaries between ML predictions and deterministic safety/environmental rules.
- **Ready for Dashboard Presentation**: Local waterfall plots, global rankings, and unified explanation schemas are fully packaged for Phase 13 Streamlit visualization.
